# 13 — Common and Differential Phase Coordinates

**Repo:** `decoherence-free-sensing`  
**Notebook role:** geometry layer  
**Previous:** `07_dfs_design_rule.ipynb`  
**Next:** `17_common_mode_rejection.ipynb`

The sensing problem becomes geometric once the phase coordinates are rotated.

Instead of treating the node phases separately,

\[
(\phi_A,\phi_B),
\]

we rotate into the coordinates that matter for differential sensing:

\[
\Phi=\frac{\phi_A+\phi_B}{2},
\qquad
\varphi=\frac{\phi_A-\phi_B}{2}.
\]

Equivalently,

\[
\phi_A=\Phi+\varphi,
\qquad
\phi_B=\Phi-\varphi.
\]

## Engineering statement

Differential sensing works by separating the coordinate to reject from the coordinate to estimate.

The common coordinate \(\Phi\) should be rejected.  
The differential coordinate \(\varphi\) should be preserved and estimated.


## Notebook outputs

Running this notebook creates:

- `results/figures/13_phase_coordinate_rotation_v2.png`
- `results/figures/13_common_differential_axes_v2.png`
- `results/figures/13_noise_projection_geometry_v2.png`
- `results/figures/13_differential_projection_rejects_common_noise_v2.png`
- `results/figures/13_coordinate_design_rule_v2.png`
- `results/phase_coordinate_summary.csv`
- `results/phase_coordinate_summary.json`
- `results/decoherence_free_sensing_phase_coordinates_outputs.zip`


In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

ROOT = Path.cwd()
RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures"
SITE = ROOT / "site"

for path in [RESULTS, FIGURES, SITE]:
    path.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 220,
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 12,
    "legend.fontsize": 11,
})


## 1. Phase coordinate rotation

The node phases can be rewritten as a common coordinate and a differential coordinate:

\[
\begin{bmatrix}
\Phi \\
\varphi
\end{bmatrix}
=
\frac{1}{2}
\begin{bmatrix}
1 & 1 \\
1 & -1
\end{bmatrix}
\begin{bmatrix}
\phi_A \\
\phi_B
\end{bmatrix}.
\]

The inverse transform is

\[
\begin{bmatrix}
\phi_A \\
\phi_B
\end{bmatrix}
=
\begin{bmatrix}
1 & 1 \\
1 & -1
\end{bmatrix}
\begin{bmatrix}
\Phi \\
\varphi
\end{bmatrix}.
\]

This is the classical coordinate version of the DFS sensing rule:

\[
J_z^+ \leftrightarrow \Phi,
\qquad
J_z^- \leftrightarrow \varphi.
\]


In [ ]:
def draw_box(ax, xy, width, height, title, body=None, fontsize=15):
    x, y = xy
    box = FancyBboxPatch(
        (x, y), width, height,
        boxstyle="round,pad=0.025,rounding_size=0.04",
        linewidth=1.8,
        edgecolor="black",
        facecolor="white"
    )
    ax.add_patch(box)
    ax.text(
        x + width/2,
        y + height*0.62 if body else y + height/2,
        title,
        ha="center",
        va="center",
        fontsize=fontsize,
        fontweight="bold"
    )
    if body:
        ax.text(
            x + width/2,
            y + height*0.28,
            body,
            ha="center",
            va="center",
            fontsize=11
        )

def draw_arrow(ax, p0, p1):
    ax.add_patch(FancyArrowPatch(
        p0, p1,
        arrowstyle="-|>",
        mutation_scale=18,
        linewidth=1.8
    ))

fig, ax = plt.subplots(figsize=(12.5, 5.5))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.text(0.5, 0.92, "Phase Coordinate Rotation", ha="center", va="center", fontsize=23)

draw_box(ax, (0.06, 0.46), 0.25, 0.26, "node phases", r"$\phi_A,\ \phi_B$")
draw_box(
    ax,
    (0.385, 0.43),
    0.23,
    0.32,
    "rotate axes",
    r"$\Phi=(\phi_A+\phi_B)/2$" + "\n" + r"$\varphi=(\phi_A-\phi_B)/2$",
    fontsize=14
)
draw_box(ax, (0.69, 0.46), 0.25, 0.26, "sensing coordinates", "reject Φ\nestimate φ")

draw_arrow(ax, (0.31, 0.59), (0.38, 0.59))
draw_arrow(ax, (0.615, 0.59), (0.685, 0.59))

ax.text(
    0.5,
    0.23,
    "Common-mode noise moves both nodes together; the differential coordinate measures separation.",
    ha="center",
    va="center",
    fontsize=14
)

path = FIGURES / "13_phase_coordinate_rotation_v2.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 2. Common and differential axes

In the original \((\phi_A,\phi_B)\) plane:

- common-mode motion follows the diagonal direction \((1,1)\),
- differential motion follows the anti-diagonal direction \((1,-1)\).

The coordinate transform is not cosmetic. It separates the nuisance axis from the sensing axis.


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 7.2))

limit = 3.2
ax.set_xlim(-limit, limit)
ax.set_ylim(-limit, limit)

x = np.linspace(-limit, limit, 300)

# Constant common phase Phi = c: phiB = 2c - phiA
# Constant differential phase varphi = c: phiB = phiA - 2c
for c in np.linspace(-2.5, 2.5, 6):
    ax.plot(x, 2*c - x, linewidth=0.9, alpha=0.35)
    ax.plot(x, x - 2*c, linestyle="--", linewidth=0.9, alpha=0.35)

ax.arrow(0, 0, 2.1, 2.1, head_width=0.12, length_includes_head=True, linewidth=2.2)
ax.arrow(0, 0, 2.1, -2.1, head_width=0.12, length_includes_head=True, linewidth=2.2)

ax.text(2.25, 2.12, r"common axis $\Phi$", fontsize=14, fontweight="bold")
ax.text(2.22, -2.25, r"differential axis $\varphi$", fontsize=14, fontweight="bold")

ax.set_xlabel(r"node A phase $\phi_A$")
ax.set_ylabel(r"node B phase $\phi_B$")
ax.set_title("Common and Differential Axes")
ax.grid(True, alpha=0.25)
ax.set_aspect("equal", adjustable="box")

ax.text(
    -3.05,
    2.55,
    "solid: constant common phase\n"
    "dashed: constant differential phase",
    fontsize=11,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="black", alpha=0.9)
)

path = FIGURES / "13_common_differential_axes_v2.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 3. Noise projection geometry

A shared noise process changes both nodes in the same direction:

\[
(\phi_A,\phi_B) \mapsto
(\phi_A+\eta,\phi_B+\eta).
\]

That means:

\[
\Phi \mapsto \Phi+\eta,
\qquad
\varphi \mapsto \varphi.
\]

The differential coordinate is unchanged by common-mode displacement.


In [ ]:
rng = np.random.default_rng(13)

n = 700
true_varphi = 0.35
eta = rng.normal(0, 1.1, n)
local = rng.normal(0, 0.06, (n, 2))

phi_A = eta + true_varphi + local[:, 0]
phi_B = eta - true_varphi + local[:, 1]

Phi = (phi_A + phi_B) / 2
varphi = (phi_A - phi_B) / 2

fig, ax = plt.subplots(figsize=(8.0, 6.5))
ax.scatter(phi_A, phi_B, s=14, alpha=0.35)

t = np.linspace(-3.4, 3.4, 200)
ax.plot(t, t, linewidth=1.5, alpha=0.7, label="common-mode direction")
ax.plot(t, t - 2*true_varphi, linestyle="--", linewidth=1.5, alpha=0.8, label="constant differential phase")

ax.set_title("Shared Noise Projects Onto the Common Coordinate")
ax.set_xlabel("observed node A phase")
ax.set_ylabel("observed node B phase")
ax.grid(True, alpha=0.3)
ax.legend()

ax.text(
    -3.15,
    2.35,
    "shared noise\nspreads along Φ\n\nsignal\nremains in φ",
    fontsize=12,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="black", alpha=0.9)
)

path = FIGURES / "13_noise_projection_geometry_v2.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 4. Differential projection rejects common noise

The coordinate transform makes the rejection rule visible.

A large common fluctuation can make the node phases look noisy. But when we project onto the differential coordinate,

\[
\varphi=\frac{\phi_A-\phi_B}{2},
\]

the shared part cancels.


In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 5.8))

ax.hist(Phi, bins=42, alpha=0.62, label=r"common coordinate $\Phi$")
ax.hist(varphi, bins=42, alpha=0.88, label=r"differential coordinate $\varphi$")
ax.axvline(true_varphi, linestyle="--", linewidth=1.8, label=r"true $\varphi$")

ax.set_title("Differential Projection Rejects Common Noise")
ax.set_xlabel("coordinate value")
ax.set_ylabel("count")
ax.grid(True, alpha=0.25)
ax.legend()

path = FIGURES / "13_differential_projection_rejects_common_noise_v2.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 5. Coordinate design rule

The phase-coordinate design rule is simple:

1. Identify the shared-noise axis.
2. Rotate coordinates into common and differential parts.
3. Reject or protect against the common coordinate.
4. Estimate the differential coordinate.

For the quantum sensor, the same rule becomes the DFS constraint:

\[
J_z^+ \leftrightarrow \Phi,
\qquad
J_z^- \leftrightarrow \varphi.
\]


In [ ]:
fig, ax = plt.subplots(figsize=(13.0, 5.8))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.text(0.5, 0.92, "Coordinate Design Rule", ha="center", va="center", fontsize=23)

items = [
    (0.05, "Identify", "shared-noise\naxis"),
    (0.285, "Rotate", "common +\ndifferential\ncoordinates"),
    (0.52, "Reject", "common coordinate\n$\\Phi$ / $J_z^+$"),
    (0.755, "Estimate", "differential coordinate\n$\\varphi$ / $J_z^-$"),
]
w, h, y = 0.19, 0.32, 0.42

for x0, title, body in items:
    draw_box(ax, (x0, y), w, h, title, body)

for x0 in [0.245, 0.48, 0.715]:
    draw_arrow(ax, (x0, y + h/2), (x0 + 0.035, y + h/2))

ax.text(
    0.5,
    0.22,
    "Separate the coordinate to reject from the coordinate to estimate.",
    ha="center",
    va="center",
    fontsize=14
)

path = FIGURES / "13_coordinate_design_rule_v2.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 6. Context summary table


In [ ]:
summary = pd.DataFrame([
    {
        "item": "common_coordinate",
        "value": "Phi = (phi_A + phi_B) / 2",
        "engineering_role": "coordinate to reject or make physically unobservable"
    },
    {
        "item": "differential_coordinate",
        "value": "varphi = (phi_A - phi_B) / 2",
        "engineering_role": "coordinate to estimate"
    },
    {
        "item": "common_axis",
        "value": "(1, 1)",
        "engineering_role": "shared-noise displacement direction"
    },
    {
        "item": "differential_axis",
        "value": "(1, -1)",
        "engineering_role": "node-separation direction"
    },
    {
        "item": "quantum_common_generator",
        "value": "J_z^+ = J_z^A + J_z^B",
        "engineering_role": "DFS constraint operator"
    },
    {
        "item": "quantum_differential_generator",
        "value": "J_z^- = J_z^A - J_z^B",
        "engineering_role": "signal generator"
    },
    {
        "item": "design_rule",
        "value": "Reject Phi while preserving varphi",
        "engineering_role": "coordinate version of DFS sensing"
    },
])

summary_csv = RESULTS / "phase_coordinate_summary.csv"
summary_json = RESULTS / "phase_coordinate_summary.json"

summary.to_csv(summary_csv, index=False)
summary.to_json(summary_json, orient="records", indent=2)

summary


## 7. Export notebook outputs

This cell creates one zip archive containing all generated figures and summary files.


In [ ]:
zip_path = RESULTS / "decoherence_free_sensing_phase_coordinates_outputs.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file in sorted(FIGURES.glob("13_*_v2.png")):
        zf.write(file, file.relative_to(ROOT))
    zf.write(summary_csv, summary_csv.relative_to(ROOT))
    zf.write(summary_json, summary_json.relative_to(ROOT))

print(f"Created: {zip_path}")

# Optional Colab download:
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass

zip_path


## 8. Next notebook

Suggested next notebook:

`17_common_mode_rejection.ipynb`

Core goal:

- simulate common-mode noise explicitly,
- compare unprotected phase readout vs differential/DFS-compatible readout,
- show why the shared-noise coordinate must be removed before entanglement optimization.
